# 16K head on the full ADI-20 dev set (Colab)

This notebook scores the frozen-AraBEST **16K-label head** (seeds 490–492) on **all 10,806 ADI-20 dev clips**. Earlier runs scored it only on the 200-clip dev pilot, where it averaged 89.83%. Full dev keeps ADI-20's uneven class sizes, from 1,649 TUN clips down to 197 SUD clips. The notebook therefore reports **balanced accuracy and macro-F1** next to accuracy. It also scores the **10,606 clips outside the pilot** separately, because the pilot informed earlier model choices.

**Before you run**

1. *Runtime → Change runtime type* → **A100** or **L4**. A T4 works, but encoding is slower.
2. Put **`adi20_8k_vs_16k_results.zip`** in `MyDrive/Arabic-Dialect-Identification-data/colab_inputs/`. This ZIP is the Kaggle output of `notebooks/pilots/ADI20_8K_vs_16K_Kaggle.ipynb`, and it holds the three trained 16K heads. An extracted copy of the folder also works.
   - If you don't have that ZIP, put **`ADI20_8K_16K_Training_Inputs.zip`** there instead. The notebook then retrains the heads with the original recipe. Retraining also means encoding the 16,000 training clips.
3. If Hugging Face asks for access to `UBC-NLP/NADI_2026_ADI20_micro`, add your token as the Colab secret **`HF_TOKEN`** (key icon in the left bar).
4. *Runtime → Run all*, then approve the Drive prompt. Keep the tab open while it encodes.

**Resuming.** Features go to `adi20_research_cache` on Drive, which is the cache the AraBEST research notebook also uses. After a disconnect, run all again. The notebook encodes only the clips that are still missing, and it prints a time estimate after the first 200 new clips.

**Output.** On Drive at `colab_runs/adi20_16k_full_dev_<time>.zip`:

- metrics for each seed, plus the mean ± SD across seeds
- scores for each dialect
- confusion matrices
- predictions and logits
- a check against the archived 200-clip predictions

## 1. Settings, Drive and pinned source

In [ ]:
from pathlib import Path
import importlib
import importlib.util
import os
import subprocess
import sys

SEEDS = (490, 491, 492)
MAX_EXTRACT_HOURS = 11        # Checked between blocks. Run all again to resume from the Drive cache.
ENCODE_BATCH = 0              # 0 = automatic: 4 on T4, 8 on L4, 16 on A100. Long clips always get smaller batches.
PREFLIGHT_ONLY = False        # True: verify inputs and print cache coverage, then stop before encoding.
HEADS_SOURCE = ""             # Optional path to adi20_8k_vs_16k_results.zip or its extracted folder.
RETRAIN_IF_NO_HEADS = True    # Without saved heads, retrain the 16K recipe from train_scaling_16000.jsonl.
SCALING_RESULTS_PACKAGE = "adi20_8k_vs_16k_results.zip"

from google.colab import drive

DRIVE_PROJECT = Path("/content/drive/MyDrive/Arabic-Dialect-Identification-data")
EXTRA_CACHE_DIRS = ()         # Optional absolute paths to more feature cache folders.

drive.mount("/content/drive")
INPUT = DRIVE_PROJECT / "colab_inputs"
WORK = DRIVE_PROJECT / "colab_runs"
CACHE_ROOT = DRIVE_PROJECT / "adi20_research_cache"   # Shared with the AraBEST research notebook.
FEATURE_INPUT_ROOTS = (INPUT, DRIVE_PROJECT / "output" / "features",
                       DRIVE_PROJECT / "reusable_adi_cache", *(Path(p) for p in EXTRA_CACHE_DIRS))
REPO = Path("/tmp/adi-research-repo")
COMMIT = "b64437a37e7eda2f1721a56860278640bf085bdd"   # Encoder and head code of the 8K/16K runs.
REPO_URL = "https://github.com/alizbeeb5/Arabic-Dialect-Identification.git"

for folder in (INPUT, WORK, CACHE_ROOT):
    folder.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
os.environ.setdefault("HF_HUB_DISABLE_XET", "1")
os.environ.setdefault("HF_HUB_DOWNLOAD_TIMEOUT", "60")   # Default is 10 s. The 1.2 GB checkpoint once timed out.
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")   # Less fragmentation on long clips.


def install_missing(packages):
    needed = [pkg for name, pkg in packages.items() if importlib.util.find_spec(name) is None]
    if needed:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *needed], check=True)
        importlib.invalidate_caches()


def git(*args, **kwargs):
    return subprocess.run(["git", "-C", str(REPO), *args], **kwargs)


REPO.mkdir(parents=True, exist_ok=True)
if not (REPO / ".git").exists():
    git("init", "--quiet", check=True)
if git("cat-file", "-e", f"{COMMIT}^{{commit}}", stderr=subprocess.DEVNULL).returncode:
    git("fetch", "--quiet", "--depth=1", REPO_URL, COMMIT, check=True)
git("checkout", "--quiet", "--detach", COMMIT, check=True)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
install_missing({"soundfile": "soundfile", "pyarrow": "pyarrow", "huggingface_hub": "huggingface_hub",
                 "sklearn": "scikit-learn"})
print("Pinned source:", COMMIT[:12])

## 2. Load and verify the dev manifests

In [ ]:
import gc
import hashlib
import io
import json
import shutil
import time
from collections import Counter, defaultdict
from datetime import datetime, timezone
from importlib.metadata import PackageNotFoundError, version
from zipfile import BadZipFile, ZipFile

import numpy as np
import pandas as pd
import soundfile as sf
import torch
import torch.nn.functional as F
from IPython.display import display
from sklearn.metrics import confusion_matrix, f1_score, precision_recall_fscore_support

from src.experiments import acoustic_classifier as clf, layer_mixture as lm
from src.experiments.casablanca_check import diagnostics
from src.models import acoustic_encoder as ae


class Stop(Exception):
    """Stops Run all with an explanation."""


torch.set_num_threads(2)
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
GPU = torch.cuda.get_device_name(DEVICE) if DEVICE.type == "cuda" else "none"
GPU_GB = torch.cuda.get_device_properties(DEVICE).total_memory / 1024**3 if DEVICE.type == "cuda" else 0
if ENCODE_BATCH <= 0:
    ENCODE_BATCH = 16 if GPU_GB >= 38 else 8 if GPU_GB >= 20 else 4
# Attention memory grows with batch size × duration². One 212 s clip needs about 9 GB, so the batch budget
# (clips × longest duration², in s²) uses roughly half the GPU. Longer clips get smaller batches and the
# longest go alone.
BATCH_BUDGET_S2 = 2500 * GPU_GB
OOM_ERRORS = tuple({torch.cuda.OutOfMemoryError, getattr(torch, "OutOfMemoryError", torch.cuda.OutOfMemoryError)})
print(f"GPU: {GPU} ({GPU_GB:.0f} GB) · up to {ENCODE_BATCH} clips per batch · "
      f"a {(BATCH_BUDGET_S2 / ENCODE_BATCH) ** .5:.0f} s clip still fits a full batch")

CONFIG = json.loads((REPO / "configs/layer_mixture.json").read_text())
OPT = CONFIG["optimization"]
LABELS = json.loads((REPO / "configs/labels.json").read_text())["label_order"]
LABEL_ID = {label: i for i, label in enumerate(LABELS)}
JOR = LABEL_ID["JOR"]
UPDATES = 1000
FEATURE_CONFIG = clf.feature_config(CONFIG["encoder"])
PROVENANCE = json.loads(json.dumps(lm.expected_provenance(FEATURE_CONFIG)))
LAYERS = PROVENANCE["layer_names"]
WIDTH = PROVENANCE["output_dim"]
LEGACY_NAME = PROVENANCE["feature_config_sha256"][:16]
HASHES = {
    "train": "56a75aa929e2a2c4592012954386cb3d91ff7f256a7004ebe7c4a776573ea4ba",
    "dev": "525ecea4b39c504b28858ddbfaf0a954af703c91ae8edddb252a0a6f2c8d138b",
    "dev_pilot": "fb8684c35ef68e25834d5a4c82cddbd31044a899a63b665c41d3f9a179052d02",
    "casablanca_validation": "5ed7a36491ba6a41782302e1f3d31c3033b4d72ac18f1f7a2d04e85177dd064e",
    "train_scaling_8000": "c3e90a5441c1d70596c635c0a5e76147c88101b1498c22e76f167090fcea842d",
    "train_scaling_16000": "19063cdde6e28cc96da4d828c08eeb37b62ed8d5f698840fb8340107ddd4b820",
}
ADI_SOURCE = "UBC-NLP/NADI_2026_ADI20_micro"
ADI_REV = "c8da454c9f474bb075d3fe608940ee29e8ccfa43"
TAG = datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%S")
OUT = WORK / f"adi20_16k_full_dev_{TAG}"


def read_jsonl(blob, name):
    if hashlib.sha256(blob).hexdigest() != HASHES[name]:
        raise ValueError(f"Checksum mismatch for {name}.jsonl")
    return [json.loads(line) for line in blob.splitlines() if line.strip()]


DEV = read_jsonl((REPO / "src/data/manifests/dev.jsonl").read_bytes(), "dev")
PILOT = read_jsonl((REPO / "data/manifests/dev_pilot.jsonl").read_bytes(), "dev_pilot")
DEV_BY_ID = {row["clip_id"]: row for row in DEV}
assert len(DEV) == len(DEV_BY_ID) == 10806 and len(PILOT) == 200
assert all(DEV_BY_ID.get(row["clip_id"]) == row for row in PILOT), "The 200-clip pilot must be a subset of full dev."
assert all(row["split"] == "dev" and row["source_dataset"] == ADI_SOURCE and row["source_revision"] == ADI_REV
           and row.get("preparation_status", "ok") == "ok" for row in DEV)
assert {row["label"] for row in DEV} == set(LABELS)
PILOT_IDS = {row["clip_id"] for row in PILOT}
IN_PILOT = np.array([row["clip_id"] in PILOT_IDS for row in DEV])
GOLD = np.array([LABEL_ID[row["label"]] for row in DEV])

# Every 16K training clip comes from the ADI-20 train split, so no exact overlap with the full train split
# rules out exact overlap with the 16K heads' training data. Speaker overlap stays unverified.
TRAIN_ALL = read_jsonl((REPO / "src/data/manifests/train.jsonl").read_bytes(), "train")
OVERLAP = {}
for field in ("clip_id", "audio_sha256", "pcm_sha256"):
    seen = {r[field] for r in TRAIN_ALL if r.get(field)}
    OVERLAP[field] = sum(row.get(field) in seen for row in DEV)
assert not any(OVERLAP.values()), f"Full dev shares audio with ADI-20 train: {OVERLAP}"
del TRAIN_ALL

print(f"Full ADI-20 dev: {len(DEV):,} clips, {sum(r['duration_sec'] for r in DEV) / 3600:.1f} h of audio, "
      f"{IN_PILOT.sum()} of them in the 200-clip pilot. Exact train/dev overlap: none.")
display(pd.Series(Counter(r["label"] for r in DEV)).reindex(LABELS).rename("clips").to_frame().T)

## 3. Find the 16K heads

The notebook looks for the saved heads first. It checks that their protocol matches the pinned commit, encoder, update budget and seeds. If they are missing, it falls back to the original training manifest and retrains in step 5.

In [ ]:
HEAD_MARKER = "train_16000/zscore_stats.npz"


def open_bundle(source):
    """Return read(relative_path) -> bytes for the results ZIP or its extracted folder."""
    source = Path(source)
    if source.is_dir():
        base = next((p.parent.parent for p in sorted(source.rglob("zscore_stats.npz"))
                     if p.parent.name == "train_16000" and (p.parent.parent / "protocol.json").is_file()), None)
        if base is None:
            raise FileNotFoundError(f"No {HEAD_MARKER} next to protocol.json under {source}")
        return lambda rel: (base / rel).read_bytes()
    archive = ZipFile(source)
    names = set(archive.namelist())
    prefix = next((n[:-len(HEAD_MARKER)] for n in sorted(names, key=len)
                   if n.endswith(HEAD_MARKER) and n[:-len(HEAD_MARKER)] + "protocol.json" in names), None)
    if prefix is None:
        raise FileNotFoundError(f"{source.name} has no {HEAD_MARKER} next to protocol.json")
    return lambda rel: archive.read(prefix + rel)


def find_heads_source():
    if HEADS_SOURCE:
        return Path(HEADS_SOURCE)
    found = [*sorted(INPUT.rglob(SCALING_RESULTS_PACKAGE)), *sorted(DRIVE_PROJECT.glob(SCALING_RESULTS_PACKAGE))]
    found += sorted(p.parent.parent for p in INPUT.rglob("zscore_stats.npz")
                    if p.parent.name == "train_16000" and (p.parent.parent / "protocol.json").is_file())
    return found[0] if found else None


def make_head(ckpt, mean, std, stat_layers):
    if list(stat_layers) != LAYERS:
        raise ValueError("The normalization statistics use a different layer list.")
    state = ckpt["state_dict"]
    if "layer_logits" not in state:
        raise ValueError("Expected a learned layer-mixture head.")
    layers = [LAYERS.index(name) for name in ckpt["layers"]]
    if state["layer_logits"].numel() != len(layers):
        raise ValueError("The head's layer weights do not match its layer list.")
    model = lm.build_model((1, len(layers), WIDTH), len(LABELS), 0)
    model.load_state_dict(state)
    model.eval()
    return {"model": model, "layers": layers, "mean": mean[layers], "std": std[layers]}


def load_saved_heads(source):
    read = open_bundle(source)
    protocol = json.loads(read("protocol.json"))
    problems = [key for key, want in (("repo_commit", COMMIT), ("encoder", CONFIG["encoder"]),
                                      ("optimizer_updates_per_head", UPDATES), ("seeds", list(SEEDS)))
                if protocol.get(key) != want]
    problems += [f"manifest {name}" for name, digest in protocol.get("manifest_sha256", {}).items()
                 if name in HASHES and HASHES[name] != digest]
    if problems:
        raise ValueError(f"{source} does not match the 16K recipe: {', '.join(problems)}")
    with np.load(io.BytesIO(read(HEAD_MARKER)), allow_pickle=False) as data:
        mean, std = data["mean"].astype(np.float32), data["std"].astype(np.float32)
        stat_layers = [str(name) for name in data["layer_names"]]
    heads = {}
    for seed in SEEDS:
        ckpt = torch.load(io.BytesIO(read(f"train_16000/head_{seed}.pt")), map_location="cpu", weights_only=True)
        if (ckpt["seed"], ckpt["train_size"], ckpt["updates"], list(ckpt["labels"])) != (seed, 16000, UPDATES, LABELS):
            raise ValueError(f"head_{seed}.pt is not the 16K head for seed {seed}.")
        heads[seed] = make_head(ckpt, mean, std, stat_layers)

    def optional_csv(rel):
        try:
            return pd.read_csv(io.BytesIO(read(rel)))
        except (KeyError, FileNotFoundError):
            return None

    info = {
        "origin": protocol.get("head_origin", "kaggle_8k_vs_16k_run"),
        "source": str(source),
        "source_sha256": ae.sha256_file(source) if Path(source).is_file() else None,
        "archived_results": optional_csv("results.csv"),
        "archived_predictions": {seed: optional_csv(f"train_16000/{seed}/ADI20_dev_predictions.csv") for seed in SEEDS},
    }
    return heads, info


def find_train16():
    for path in sorted(INPUT.rglob("train_scaling_16000.jsonl")):
        blob = path.read_bytes()
        if hashlib.sha256(blob).hexdigest() == HASHES["train_scaling_16000"]:
            return blob
    for path in sorted(INPUT.rglob("*.zip")):
        try:
            with ZipFile(path) as archive:
                for name in archive.namelist():
                    if name.endswith("train_scaling_16000.jsonl"):
                        blob = archive.read(name)
                        if hashlib.sha256(blob).hexdigest() == HASHES["train_scaling_16000"]:
                            return blob
        except BadZipFile:
            continue
    return None


HEADS, TRAIN16 = None, None
heads_source = find_heads_source()
if heads_source is not None:
    HEADS, HEAD_INFO = load_saved_heads(heads_source)
    first = HEADS[SEEDS[0]]
    print(f"Loaded saved 16K heads from {heads_source}")
    print(f"Each head mixes {len(first['layers'])} layers: {LAYERS[first['layers'][0]]} … {LAYERS[first['layers'][-1]]}")
else:
    blob = find_train16() if RETRAIN_IF_NO_HEADS else None
    if blob is None:
        raise Stop(f"Add {SCALING_RESULTS_PACKAGE} (Kaggle output of notebooks/pilots/ADI20_8K_vs_16K_Kaggle.ipynb) "
                   f"to {INPUT}. To retrain instead, add ADI20_8K_16K_Training_Inputs.zip there.")
    TRAIN16 = read_jsonl(blob, "train_scaling_16000")
    assert Counter(row["label"] for row in TRAIN16) == dict.fromkeys(LABELS, 800)
    assert all(row["split"] == "train" and row["source_dataset"] == ADI_SOURCE
               and row["source_revision"] == ADI_REV for row in TRAIN16)
    for field in ("clip_id", "audio_sha256", "pcm_sha256"):
        values = {row[field] for row in TRAIN16}
        assert len(values) == 16000 and values.isdisjoint(row[field] for row in DEV), f"Train/dev overlap on {field}"
    HEAD_INFO = {"origin": "retrained_in_colab", "source": "train_scaling_16000.jsonl",
                 "archived_results": None, "archived_predictions": {}}
    print("No saved 16K heads found. Step 5 retrains them from the verified 16,000-clip manifest.")

## 4. Reuse or encode AraBEST features

The features are pooled AraBEST outputs for all 26 layers, stored in FP32. The cache keys each clip by its audio hash and the full encoder recipe, so a mismatched cache is never read. The notebook also reads older Kaggle caches placed in `colab_inputs` or `reusable_adi_cache`. New parts go only to `adi20_research_cache`.

In [ ]:
FEATURE_DEADLINE = time.monotonic() + MAX_EXTRACT_HOURS * 3600
BLOCK = 64   # Clips decoded together and written as one cache part.
PAUSE = "Extraction time limit reached. Completed parts are on Drive; run all again to continue."


class ExtractionPaused(Stop):
    pass


def atomic_text(path, value):
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(value, encoding="utf-8")
    tmp.replace(path)


class FeatureBank:
    """Pooled features in the research notebook's 'mean' cache format. Also reads older Kaggle caches."""

    def __init__(self):
        self.spec = {"encoder": PROVENANCE, "mode": "mean", "frames": False, "format": 1}
        self.spec_hash = hashlib.sha256(ae.canonical_json(self.spec)).hexdigest()[:20]
        self.root = CACHE_ROOT / "mean" / self.spec_hash
        self.root.mkdir(parents=True, exist_ok=True)
        atomic_text(self.root / "config.json", json.dumps(self.spec))
        self.index, self.sources = {}, Counter()
        mounted = {c for root in FEATURE_INPUT_ROOTS if root.is_dir() for c in root.rglob("config.json")}
        for config in [self.root / "config.json", *sorted(mounted - {self.root / "config.json"})]:
            self._mount(config)

    def _mount(self, config):
        folder = config.parent
        if folder.name not in {LEGACY_NAME, self.spec_hash}:
            return
        stored = json.loads(config.read_text())
        if folder.name == LEGACY_NAME:
            if any(stored.get(key) != PROVENANCE[key] for key in ae.FeatureCache.IDENTITY_FIELDS):
                return
        elif stored != self.spec:
            return
        index_paths = sorted(folder.glob("part-*.index.jsonl"))
        if (folder / "index.jsonl").is_file():
            index_paths.insert(0, folder / "index.jsonl")
        parts = {part.name for part in folder.glob("part-*.npz")}
        for path in index_paths:
            for line in path.read_text().splitlines():
                if not line.strip():
                    continue
                item = json.loads(line)
                if item["part"] in parts and item["clip_id"] not in self.index:
                    self.index[item["clip_id"]] = (folder / item["part"], item)
                    self.sources[str(folder)] += 1

    def key(self, row):
        return ae.clip_cache_key(row["clip_id"], row["audio_sha256"], PROVENANCE)

    def has(self, row):
        entry = self.index.get(row["clip_id"])
        return bool(entry and entry[1]["cache_key"] == self.key(row))

    def missing(self, rows):
        return [row for row in rows if not self.has(row)]

    def write(self, rows, values, seconds):
        ids = [row["clip_id"] for row in rows]
        keys = [self.key(row) for row in rows]
        stem = "part-" + hashlib.sha256("".join(keys).encode()).hexdigest()[:24]
        part, temp = self.root / f"{stem}.npz", self.root / f"{stem}.tmp"
        with temp.open("wb") as stream:
            np.savez(stream, clip_ids=ids, cache_keys=keys, features=np.stack(values).astype(np.float32))
        temp.replace(part)
        items = [{"clip_id": cid, "part": part.name, "row": i, "cache_key": key,
                  "seconds": seconds / len(ids), "session": TAG} for i, (cid, key) in enumerate(zip(ids, keys))]
        atomic_text(self.root / f"{stem}.index.jsonl", "".join(json.dumps(item) + "\n" for item in items))
        self.index.update({item["clip_id"]: (part, item) for item in items})

    def load(self, rows):
        result = np.empty((len(rows), len(LAYERS), WIDTH), np.float32)
        parts = defaultdict(list)
        for position, row in enumerate(rows):
            if not self.has(row):
                raise ValueError(f"Feature cache is missing {row['clip_id']}.")
            part, item = self.index[row["clip_id"]]
            parts[part].append((position, item))
        for part, entries in parts.items():
            with np.load(part, allow_pickle=False) as data:
                values, ids, keys = data["features"], data["clip_ids"], data["cache_keys"]
                for position, item in entries:
                    i = item["row"]
                    if ids[i] != item["clip_id"] or keys[i] != item["cache_key"]:
                        raise ValueError(f"Wrong feature row in {part}")
                    result[position] = values[i]
        if not np.isfinite(result).all():
            raise ValueError("Nonfinite cached features.")
        return result


ENCODER = None


def hf_token():
    if os.environ.get("HF_TOKEN"):
        return os.environ["HF_TOKEN"]
    try:
        from google.colab import userdata
        return userdata.get("HF_TOKEN")
    except Exception:
        return None


def drop_attention_map(module, inputs, output):
    """SpeechBrain's ConformerEncoder keeps every layer's attention map until the forward pass ends. The
    features never use them, and for a 212 s clip the 24 maps take about 22 GB. Dropping each map when its
    layer returns leaves the hidden states, and so the cached features, unchanged."""
    return (output[0], None) if isinstance(output, tuple) else output


def install_attention_hooks(encoder):
    attention = [m for m in encoder.modules() if type(m).__name__ == "RelPosMHAXL"]
    if len(attention) != encoder.architecture.num_layers:
        raise RuntimeError(f"Expected {encoder.architecture.num_layers} attention modules, found {len(attention)}.")
    for module in attention:
        module.register_forward_hook(drop_attention_map)


def get_encoder():
    global ENCODER
    if ENCODER is None:
        if DEVICE.type != "cuda":
            raise Stop("Encoding needs a GPU. Runtime → Change runtime type → A100, L4 or T4.")
        token = hf_token()
        if token:
            os.environ["HF_TOKEN"] = token
        try:
            installed = version("speechbrain")
        except PackageNotFoundError:
            installed = ""
        if installed != "1.1.2":
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "speechbrain==1.1.2"],
                           check=True)
        install_missing({"hyperpyyaml": "hyperpyyaml", "sentencepiece": "sentencepiece",
                         "joblib": "joblib", "tqdm": "tqdm"})
        importlib.invalidate_caches()
        ENCODER = ae.AcousticEncoder.from_pretrained(FEATURE_CONFIG, device=str(DEVICE), cache_dir="/tmp/adi-hf")
        install_attention_hooks(ENCODER)
        clf.frozen_summary(ENCODER)
        if json.loads(json.dumps(ENCODER.provenance())) != PROVENANCE:
            raise ValueError("The loaded encoder differs from the cache recipe.")
    return ENCODER


def wave_from_bytes(row, payload):
    if not payload or hashlib.sha256(payload).hexdigest() != row["audio_sha256"]:
        raise ValueError(f"Audio hash mismatch for {row['clip_id']}.")
    wave, rate = sf.read(io.BytesIO(payload), dtype="float32", always_2d=True)
    return ae.prepare_waveform(wave.mean(axis=1), rate, FEATURE_CONFIG)


def encode(waves):
    with torch.inference_mode():
        encoder = get_encoder()
        hidden, mask, _ = encoder.encode(waves)
        pooled = encoder.pool(hidden, mask).cpu().numpy()
    if not np.isfinite(pooled).all():
        raise FloatingPointError("AraBEST returned nonfinite features.")
    return list(pooled)


def parquet_waves(rows):
    """Yield lists of (row, waveform) read from the pinned Hugging Face parquet shards."""
    import pyarrow.parquet as pq
    from huggingface_hub import HfFileSystem, hf_hub_download
    from tempfile import TemporaryDirectory

    token = hf_token()
    hub = HfFileSystem(token=token)
    shards = defaultdict(list)
    for row in rows:
        if (row["source_dataset"], row["source_revision"]) != (ADI_SOURCE, ADI_REV):
            raise ValueError(f"Unexpected source for {row['clip_id']}")
        shards[row["source_shard"]].append(row)
    for number, (shard, members) in enumerate(sorted(shards.items()), 1):
        if time.monotonic() >= FEATURE_DEADLINE:
            raise ExtractionPaused(PAUSE)
        print(f"Shard {number}/{len(shards)} · {len(members):,} clips · {shard}", flush=True)
        positions = {int(row["source_row"]): row for row in members}
        if len(positions) != len(members):
            raise ValueError(f"Duplicate source rows in {shard}")
        with TemporaryDirectory(prefix="adi-parquet-") as tmp:
            if "validation" in shard:   # Full dev needs every row, so download the whole shard.
                reader = open(hf_hub_download(ADI_SOURCE, shard, repo_type="dataset", revision=ADI_REV,
                                              token=token, local_dir=tmp), "rb")
            else:                       # Training clips are sparse within each shard.
                reader = hub.open(f"datasets/{ADI_SOURCE}@{ADI_REV}/{shard}", "rb", block_size=8 * 1024**2)
            with reader:
                parquet = pq.ParquetFile(reader)
                offset = 0
                for group in range(parquet.num_row_groups):
                    stop = offset + parquet.metadata.row_group(group).num_rows
                    needed = sorted(i for i in positions if offset <= i < stop)
                    cursor = offset
                    if needed:
                        for batch in parquet.iter_batches(row_groups=[group], columns=["audio"], batch_size=BLOCK):
                            if time.monotonic() >= FEATURE_DEADLINE:
                                raise ExtractionPaused(PAUSE)
                            end = cursor + len(batch)
                            audio = batch.column(0).field("bytes")
                            block = [positions.pop(i) for i in needed if cursor <= i < end]
                            if block:
                                yield [(row, wave_from_bytes(row, audio[row["source_row"] - cursor].as_py()))
                                       for row in block]
                            cursor = end
                            if cursor > needed[-1]:
                                break
                    offset = stop
                    if not positions:
                        break
        if positions:
            raise ValueError(f"{len(positions)} manifest rows were not found in {shard}.")


def gpu_batches(block):
    """Split a length-sorted block so clips × (longest seconds)² stays within BATCH_BUDGET_S2."""
    batch = []
    for item in block:
        seconds = item[1].numel() / FEATURE_CONFIG.sample_rate_hz   # Longest so far, because the block is sorted.
        if batch and (len(batch) == ENCODE_BATCH or (len(batch) + 1) * seconds ** 2 > BATCH_BUDGET_S2):
            yield batch
            batch = []
        batch.append(item)
    if batch:
        yield batch


def encode_safely(waves):
    try:
        return encode(waves)
    except OOM_ERRORS:
        if len(waves) == 1:
            raise
    # Outside the except block the failed batch is unreferenced. gc.collect frees its GPU tensors
    # before the one-clip retry.
    gc.collect()
    torch.cuda.empty_cache()
    return [value for wave in waves for value in encode_safely([wave])]


def ensure_features(rows, label):
    todo = BANK.missing(rows)
    if not todo:
        print(f"{label}: all {len(rows):,} clips cached")
        return
    total_audio = sum(row["duration_sec"] for row in todo)
    print(f"{label}: encoding {len(todo):,} clips ({total_audio / 3600:.1f} h of audio)", flush=True)
    get_encoder()
    started, done, done_audio, next_report = time.monotonic(), 0, 0.0, 200
    for block in parquet_waves(todo):
        block.sort(key=lambda item: item[1].numel())   # Similar lengths share a batch. Pooling ignores padding.
        tick = time.perf_counter()
        values = []
        for batch in gpu_batches(block):
            values.extend(encode_safely([wave for _, wave in batch]))
        BANK.write([row for row, _ in block], values, time.perf_counter() - tick)
        done += len(block)
        done_audio += sum(row["duration_sec"] for row, _ in block)
        if done >= next_report or done == len(todo):
            elapsed = time.monotonic() - started
            left = elapsed * (total_audio - done_audio) / max(done_audio, 1e-9) / 3600
            print(f"{label}: {done:,}/{len(todo):,} encoded in {elapsed / 60:.1f} min · about {left:.1f} h left",
                  flush=True)
            next_report = done + 1000


BANK = FeatureBank()
print("Feature cache:", BANK.root)
for folder, count in BANK.sources.most_common():
    print(f"  {count:,} indexed clips in {folder}")
dev_missing = BANK.missing(DEV)
print(f"ADI-20 dev: {len(DEV) - len(dev_missing):,}/{len(DEV):,} cached "
      f"(pilot {sum(BANK.has(row) for row in PILOT)}/200) · "
      f"{sum(row['duration_sec'] for row in dev_missing) / 3600:.1f} h of audio to encode")
if TRAIN16 is not None:
    train_missing = BANK.missing(TRAIN16)
    print(f"16K training clips: {len(TRAIN16) - len(train_missing):,}/16,000 cached · "
          f"{sum(row['duration_sec'] for row in train_missing) / 3600:.1f} h of audio to encode")

In [ ]:
if PREFLIGHT_ONLY:
    raise Stop("Preflight finished. Set PREFLIGHT_ONLY = False and run all again.")
ensure_features(DEV, "ADI-20 dev")
if TRAIN16 is not None:
    ensure_features(TRAIN16, "16K training clips")
ENCODER = None
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

## 5. Retrain the 16K heads (only without saved heads)

This step reuses the recipe from `ADI20_8K_vs_16K_Kaggle.ipynb`: z-scores fit on the 16,000 training clips, a learned mixture over all cached layers, 1,000 AdamW updates and seeds 490–492. Head training on another GPU type can differ slightly from the Kaggle weights. The retrained heads are saved, so later runs can set `HEADS_SOURCE` to them.

In [ ]:
def fit_head(features, labels, seed):
    init_seed, shuffle_seed = clf.run_seeds(seed)
    model = lm.build_model(tuple(features.shape), len(LABELS), init_seed).to(DEVICE)
    optimizer = lm.make_optimizer(model, OPT)
    features, labels = features.to(DEVICE), labels.to(DEVICE)
    generator = torch.Generator().manual_seed(shuffle_seed)
    steps = 0
    while steps < UPDATES:
        order = torch.randperm(len(features), generator=generator)
        for indices in order.split(OPT["batch_size"]):
            indices = indices.to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            loss = F.cross_entropy(model(features[indices]), labels[indices])
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), OPT["gradient_clip_norm"])
            optimizer.step()
            steps += 1
            if steps == UPDATES:
                break
    return model.cpu().eval()


if HEADS is None:
    torch.use_deterministic_algorithms(True)
    raw = BANK.load(TRAIN16)
    mean = raw.mean(axis=0)
    std = np.maximum(raw.std(axis=0), CONFIG["preprocessing"]["std_floor"])
    raw -= mean
    raw /= std
    features = torch.from_numpy(raw)
    labels = torch.tensor([LABEL_ID[row["label"]] for row in TRAIN16], dtype=torch.long)
    saved = OUT / "retrained_16k_heads"
    (saved / "train_16000").mkdir(parents=True, exist_ok=True)
    np.savez(saved / HEAD_MARKER, mean=mean, std=std, layer_names=np.array(LAYERS))
    HEADS = {}
    for seed in SEEDS:
        model = fit_head(features, labels, seed)
        ckpt = {"state_dict": model.state_dict(), "labels": LABELS, "layers": LAYERS, "seed": seed,
                "updates": UPDATES, "train_size": len(TRAIN16), "config": CONFIG}
        torch.save(ckpt, saved / "train_16000" / f"head_{seed}.pt")
        HEADS[seed] = make_head(ckpt, mean, std, LAYERS)
        print(f"Seed {seed}: trained for {UPDATES} updates", flush=True)
    (saved / "protocol.json").write_text(json.dumps({
        "repo_commit": COMMIT, "encoder": CONFIG["encoder"],
        "manifest_sha256": {"train_scaling_16000": HASHES["train_scaling_16000"]},
        "seeds": list(SEEDS), "optimizer_updates_per_head": UPDATES, "head_device": str(DEVICE),
        "gpu": GPU, "head_origin": "retrained_in_colab"}, indent=2))
    HEAD_INFO["source"] = str(saved)
    torch.use_deterministic_algorithms(False)
    del raw, features
    gc.collect()
    print(f"Saved retrained heads. To reuse them, set HEADS_SOURCE = {str(saved)!r}")
else:
    print("Using the saved 16K heads; nothing to retrain.")

## 6. Score full dev

Three cohorts are scored for each seed:

- **full dev**, 10,806 clips
- **the 200-clip pilot**, which should reproduce the archived 89.83%
- **dev without the pilot**, 10,606 clips

Macro-F1 averages over the dialects present. Balanced accuracy is the mean recall per dialect. The reproduction check compares predictions on the pilot clips with the archived Kaggle predictions. A flip on one borderline clip can come from re-encoding on a different GPU. Several flips per seed mean the features or heads do not match.

In [ ]:
X = BANK.load(DEV)
LOGITS, PRED = {}, {}
with torch.no_grad():
    for seed, head in HEADS.items():
        out = np.empty((len(DEV), len(LABELS)), np.float32)
        for start in range(0, len(DEV), 512):
            block = (X[start:start + 512][:, head["layers"]] - head["mean"]) / head["std"]
            out[start:start + 512] = head["model"](torch.from_numpy(np.ascontiguousarray(block, np.float32))).numpy()
        if not np.isfinite(out).all():
            raise FloatingPointError("Nonfinite logits")
        LOGITS[seed], PRED[seed] = out, out.argmax(1)
del X
gc.collect()

COHORTS = {"ADI20_dev_full": np.ones(len(DEV), bool), "ADI20_dev_pilot200": IN_PILOT,
           "ADI20_dev_without_pilot": ~IN_PILOT}
RATES = ["accuracy", "macro_f1", "balanced_accuracy", "jor_recall", "region_accuracy"]


def score(mask, pred):
    rows = [DEV[i] for i in np.flatnonzero(mask)]
    gold, pred = GOLD[mask], pred[mask]
    present = np.unique(gold)
    correct = gold == pred
    diag = diagnostics(rows, [LABELS[i] for i in pred], [LABELS[i] for i in present], LABELS)[0]
    return {
        "accuracy": float(correct.mean()),
        "macro_f1": float(f1_score(gold, pred, labels=present, average="macro", zero_division=0)),
        "balanced_accuracy": float(np.mean([correct[gold == c].mean() for c in present])),
        "jor_recall": float(correct[gold == JOR].mean()),
        "region_accuracy": diag["region_accuracy"],
        "n": int(mask.sum()),
    }, diag


records, per_dialect, diags = [], [], {}
for seed in SEEDS:
    for cohort, mask in COHORTS.items():
        metrics, diag = score(mask, PRED[seed])
        records.append({"arm": "train_16000", "seed": seed, "split": cohort, **metrics})
        if cohort == "ADI20_dev_full":
            diags[seed] = diag
    precision, recall, f1, support = precision_recall_fscore_support(
        GOLD, PRED[seed], labels=range(len(LABELS)), zero_division=0)
    per_dialect.append(pd.DataFrame({"seed": seed, "label": LABELS, "support": support,
                                     "precision": precision, "recall": recall, "f1": f1}))
RESULTS = pd.DataFrame(records)
SUMMARY = RESULTS.groupby("split", sort=False)[RATES].agg(["mean", "std"])
PER_DIALECT = pd.concat(per_dialect, ignore_index=True)
DIALECT_MEAN = PER_DIALECT.groupby("label", sort=False)[["support", "precision", "recall", "f1"]].mean()

for cohort in COHORTS:
    row = SUMMARY.loc[cohort]
    print(f"{cohort} ({COHORTS[cohort].sum():,} clips): accuracy {row['accuracy', 'mean']:.2%} ± "
          f"{row['accuracy', 'std']:.2%} · macro-F1 {row['macro_f1', 'mean']:.4f} · "
          f"balanced accuracy {row['balanced_accuracy', 'mean']:.2%}")
print("\nMean ± SD over seeds 490–492, percent")
display((100 * SUMMARY).round(2))
print("Per dialect on full dev, mean over seeds (sorted by recall)")
display(DIALECT_MEAN.assign(support=DIALECT_MEAN.support.astype(int))
        .sort_values("recall").style.format({"precision": "{:.1%}", "recall": "{:.1%}", "f1": "{:.3f}"}))

CONFUSION = {seed: confusion_matrix(GOLD, PRED[seed], labels=range(len(LABELS))) for seed in SEEDS}
mean_confusion = np.mean([CONFUSION[seed] for seed in SEEDS], axis=0)
np.fill_diagonal(mean_confusion, 0)
top = np.dstack(np.unravel_index(np.argsort(mean_confusion, axis=None)[::-1][:10], mean_confusion.shape))[0]
print("Most frequent confusions on full dev (mean clips over seeds)")
display(pd.DataFrame([{"gold": LABELS[g], "predicted": LABELS[p], "clips": mean_confusion[g, p],
                       "share_of_gold": mean_confusion[g, p] / (GOLD == g).sum()} for g, p in top])
        .style.format({"clips": "{:.1f}", "share_of_gold": "{:.1%}"}))

repro = []
archived = HEAD_INFO.get("archived_results")
archived_dev = (archived[(archived.arm == "train_16000") & (archived.split == "ADI20_dev")].set_index("seed")
                if archived is not None else None)
pilot_positions = np.flatnonzero(IN_PILOT)
for seed in SEEDS:
    entry = {"seed": seed, "reproduced_accuracy": RESULTS.query("seed == @seed and split == 'ADI20_dev_pilot200'")
             .accuracy.item()}
    if archived_dev is not None and seed in archived_dev.index:
        entry["archived_accuracy"] = float(archived_dev.loc[seed, "accuracy"])
    original = HEAD_INFO["archived_predictions"].get(seed)
    if original is not None:
        before = dict(zip(original.clip_id, original.prediction))
        entry["same_prediction"] = sum(before.get(DEV[i]["clip_id"]) == LABELS[PRED[seed][i]] for i in pilot_positions)
        entry["compared"] = len(pilot_positions)
    repro.append(entry)
REPRO = pd.DataFrame(repro)
print("Reproduction on the 200-clip pilot")
display(REPRO)
if "same_prediction" in REPRO and (REPRO.compared - REPRO.same_prediction).max() > 1:
    print("WARNING: more than one pilot prediction changed for a seed. Check the features and heads before "
          "reporting these numbers.")

In [ ]:
OUT.mkdir(parents=True, exist_ok=True)
RESULTS.to_csv(OUT / "results.csv", index=False)
SUMMARY.to_csv(OUT / "summary.csv")
PER_DIALECT.to_csv(OUT / "per_dialect_by_seed.csv", index=False)
DIALECT_MEAN.to_csv(OUT / "per_dialect_mean.csv")
REPRO.to_csv(OUT / "reproduction_pilot200.csv", index=False)
np.savez_compressed(OUT / "logits_full_dev.npz", clip_ids=np.array([row["clip_id"] for row in DEV]),
                    labels=np.array(LABELS), **{f"seed_{seed}": LOGITS[seed] for seed in SEEDS})
for seed in SEEDS:
    folder = OUT / "train_16000" / str(seed)
    folder.mkdir(parents=True, exist_ok=True)
    pd.DataFrame({"clip_id": [row["clip_id"] for row in DEV], "gold": [row["label"] for row in DEV],
                  "prediction": [LABELS[i] for i in PRED[seed]], "in_pilot200": IN_PILOT,
                  "duration_sec": [row["duration_sec"] for row in DEV]}).to_csv(
        folder / "ADI20_dev_full_predictions.csv", index=False)
    pd.DataFrame(CONFUSION[seed], index=LABELS, columns=LABELS).to_csv(folder / "ADI20_dev_full_confusion.csv")
    (folder / "ADI20_dev_full_diagnostics.json").write_text(json.dumps(diags[seed], indent=2))

first = HEADS[SEEDS[0]]
protocol = {
    "task": "16K-label frozen AraBEST head scored on the full ADI-20 dev split",
    "created_utc": TAG,
    "repo_commit": COMMIT,
    "encoder": CONFIG["encoder"],
    "manifest_sha256": {name: HASHES[name] for name in ("dev", "dev_pilot", "train")},
    "cohorts": {name: int(mask.sum()) for name, mask in COHORTS.items()},
    "exact_train_dev_overlap": OVERLAP,
    "heads": {"origin": HEAD_INFO["origin"], "source": HEAD_INFO["source"],
              "source_sha256": HEAD_INFO.get("source_sha256"), "seeds": list(SEEDS),
              "optimizer_updates_per_head": UPDATES, "mixture_layers": [LAYERS[i] for i in first["layers"]]},
    "feature_cache": {"root": str(BANK.root), "spec_sha256_prefix": BANK.spec_hash},
    "gpu": GPU,
    "encode_batch": ENCODE_BATCH,
    "batch_budget_s2": BATCH_BUDGET_S2,
    "metrics": "Accuracy; macro-F1 over present labels; balanced accuracy = mean per-dialect recall.",
    "speaker_independence_verified": False,
}
(OUT / "protocol.json").write_text(json.dumps(protocol, indent=2))
archive = shutil.make_archive(str(WORK / OUT.name), "zip", OUT)
print("Saved:", archive)